# P0 data audit — covariate-safe TSFM

CPU-only audit for the four frozen FEV tasks. It verifies dataset fingerprints, target/covariate schemas, rolling cutoffs, and the chronological calibration/evaluation boundary. It does **not** run a forecasting model or inspect evaluation losses.

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/covariate-safe-tsfm')
if not REPO.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/FlyMe2star/covariate-safe-tsfm.git', str(REPO)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/autogluon/fev.git@38007871dcf6dc6b04aed3a54d9cd86678d48d0b'
], check=True)

# Editable installs add a .pth file that is read only when Python starts.
# Add src explicitly so this already-running Colab kernel needs no restart.
SOURCE_ROOT = str(REPO / 'src')
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
importlib.invalidate_caches()

covsafe = importlib.import_module('covsafe')

print('Repository ready at', REPO)
print('covsafe import:', covsafe.__file__)

In [ ]:
import json
import platform
from datetime import UTC, datetime

import datasets
import fev
import yaml

from covsafe.config import canonical_config_hash
from covsafe.protocol import temporal_origin_partition

CONFIG_PATH = REPO / 'configs/diagnostic/covariate_utility_p0.yaml'
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
config_hash = canonical_config_hash(config)
assert config_hash == 'b2ace898d038', config_hash
print('Frozen config hash:', config_hash)

In [ ]:
def make_task(task_config):
    fields = {
        'dataset_path': config['data']['dataset_path'],
        'dataset_config': task_config['dataset_config'],
        'horizon': task_config['horizon'],
        'num_windows': task_config['num_windows'],
        'seasonality': task_config['seasonality'],
        'known_dynamic_columns': task_config['known_dynamic_columns'],
        'past_dynamic_columns': task_config['past_dynamic_columns'],
        'static_columns': [],
        'eval_metric': 'SQL',
        'extra_metrics': [
            'MASE',
            {'name': 'WAPE', 'epsilon': 1.0},
            {'name': 'WQL', 'epsilon': 1.0},
        ],
        'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    }
    if task_config.get('target') is not None:
        fields['target'] = task_config['target']
    if task_config.get('initial_cutoff') is not None:
        fields['initial_cutoff'] = task_config['initial_cutoff']
    return fev.Task(**fields)


def audit_task(task_config):
    task = make_task(task_config)
    full_data = task.load_full_dataset(num_proc=1)
    partition = temporal_origin_partition(
        task.num_windows,
        calibration_fraction=config['data']['calibration_fraction'],
        minimum_per_partition=config['data']['minimum_origins_per_partition'],
    )

    boundary_checks = {}
    for label, index in [('first', 0), ('last', task.num_windows - 1)]:
        window = task.get_window(index, num_proc=1)
        past_data, future_data = window.get_input_data()
        past_columns = set(past_data.column_names)
        future_columns = set(future_data.column_names)
        assert set(task.known_dynamic_columns) <= past_columns
        assert set(task.past_dynamic_columns) <= past_columns
        assert set(task.known_dynamic_columns) <= future_columns
        assert not (set(task.past_dynamic_columns) & future_columns)
        assert not (set(task.target_columns) & future_columns)
        boundary_checks[label] = {
            'window_index': index,
            'cutoff': window.cutoff,
            'past_item_count': len(past_data),
            'future_item_count': len(future_data),
            'past_columns': sorted(past_columns),
            'future_columns': sorted(future_columns),
        }

    return {
        'dataset_config': task.dataset_config,
        'dataset_fingerprint': task._dataset_fingerprint,
        'full_item_count': len(full_data),
        'frequency': task.freq,
        'target_columns': task.target_columns,
        'known_dynamic_columns': task.known_dynamic_columns,
        'past_dynamic_columns': task.past_dynamic_columns,
        'cutoffs': task.cutoffs,
        'calibration_indices': list(partition.calibration),
        'evaluation_indices': list(partition.evaluation),
        'boundary_checks': boundary_checks,
    }


task_audits = [audit_task(task_config) for task_config in config['data']['tasks']]
print(json.dumps(task_audits, indent=2, ensure_ascii=False, default=str))

In [ ]:
git_commit = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True
).strip()
manifest = {
    'audit': 'p0-fev-data-schema-and-origin-boundary',
    'schema_version': 1,
    'result_status': 'smoke_only',
    'config_hash': config_hash,
    'git_commit': git_commit,
    'official_final_test_instantiated': False,
    'created_at_utc': datetime.now(UTC).isoformat(),
    'runtime': {
        'python': sys.version,
        'platform': platform.platform(),
        'fev': getattr(fev, '__version__', 'unknown'),
        'datasets': datasets.__version__,
    },
    'tasks': task_audits,
}

output_path = REPO / 'outputs/p0/p0_data_audit.json'
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False, default=str),
    encoding='utf-8',
)
print(json.dumps(manifest, indent=2, ensure_ascii=False, default=str))
print('Local manifest:', output_path)

## Expected outcome

The last cell should report four tasks, the frozen hash `b2ace898d038`, disjoint chronological origin partitions, and `official_final_test_instantiated: false`. Send the final JSON block back before any GPU smoke run.